# `evap_cool` — equilibrium thermodynamics & figures

Visualises the post-processed equilibrium thermodynamics produced by
`evap_cool.post_processing`, for each trap and each statistic (MB / BE / FD):

- **state functions** — $\Omega,\ S,\ P,\ H,\ F,\ G$
- **thermal coefficients** — $C_V,\ C_P,\ \kappa_T,\ B_P$

Run **`evap_cool_usage.ipynb`** first; it writes the `*_bosons.json`,
`*_fermions.json`, `*_mb.json` runs and their `*_thermo.json` companions into a
timestamped session folder. This notebook loads the most recent session and
saves figures to `session/figures/`.

> These are per-trap **dimensional** plots (J for box and the two mixed traps,
> eV for harmonic and quadrupole). The cross-trap dimensionless collapse onto a
> single figure lives in a separate notebook.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from evap_cool import (
    load_run, load_thermodynamics, list_sessions, list_runs,
    process_and_save_run, process_and_save_mb_run,
    plot_state_functions, plot_thermal_coefficients,
    BoxTrap, QuadrupoleTrap, OscillatorTrap, OscBoxTrap, BoxOscTrap,
)

## Locate the most recent session

`list_sessions("runs")` returns every `runs/<date>/<time>/` folder
chronologically; we take the latest. Override `session` manually to target a
specific run.

In [ ]:
sessions = list_sessions("runs")
if not sessions:
    raise RuntimeError(
        "No sessions under runs/. Run evap_cool_usage.ipynb first to generate "
        "the runs and thermo files."
    )

session = sessions[-1]
fig_dir = session / "figures"
fig_dir.mkdir(exist_ok=True)
print(f"Using session : {session}")
print("Files present :")
for p in list_runs(session):
    print(f"  {p.relative_to(session)}")

## Trap roster

**Must match the roster in `evap_cool_usage.ipynb`** — same `key` (filename
stem), same trap parameters, same energy units. The trap instances are needed
to (re)generate any missing `*_thermo.json` on the fly.

In [ ]:
TRAPS = [
    dict(key="box",         name="Box",            units="J",
         trap=BoxTrap(V=1e-11)),
    dict(key="box2d_osc1d", name="2D box + 1D HO", units="J",
         trap=BoxOscTrap(omega_z=2*np.pi*100, Sigma=1e-8)),
    dict(key="osc2d_box1d", name="2D HO + 1D box", units="J",
         trap=OscBoxTrap(omega_x=2*np.pi*100, omega_y=2*np.pi*100, L=1e-4)),
    dict(key="oscillator",  name="Oscillator",     units="eV",
         trap=OscillatorTrap(omega=2*np.pi*100)),
    dict(key="quadrupole",  name="Quadrupole",     units="eV",
         trap=QuadrupoleTrap(A_bar=1e-15)),
]

## Helpers: load (and if needed post-process) one trap, then plot + save

`ensure_and_load` requires the three source runs (from the evap notebook) and
generates any missing `*_thermo.json` via the post-processing helpers.
`plot_trap` draws the state functions and the thermal coefficients for one trap
(BE green, FD red, MB blue underlay) and writes both figures as PDF + PNG.

In [ ]:
def ensure_and_load(session, key, trap):
    """Return (T_b, thr_b, T_f, thr_f, T_mb, thr_mb) for one trap, or None."""
    src = {s: session / f"{key}_{s}.json" for s in ("bosons", "fermions", "mb")}
    missing = [p.name for p in src.values() if not p.exists()]
    if missing:
        print(f"  [skip] {key}: missing source runs {missing}")
        return None

    thr = {s: session / f"{key}_{s}_thermo.json" for s in ("bosons", "fermions", "mb")}
    if not thr["bosons"].exists():   process_and_save_run(src["bosons"],   trap, sign=+1)
    if not thr["fermions"].exists(): process_and_save_run(src["fermions"], trap, sign=-1)
    if not thr["mb"].exists():       process_and_save_mb_run(src["mb"],    trap)

    return (
        load_run(src["bosons"])["results"]["T"],   load_thermodynamics(thr["bosons"])["results"],
        load_run(src["fermions"])["results"]["T"], load_thermodynamics(thr["fermions"])["results"],
        load_run(src["mb"])["results"]["T"],       load_thermodynamics(thr["mb"])["results"],
    )


def plot_trap(session, entry, fig_dir, *, save=True, show=True):
    """Plot + save state functions and thermal coefficients for one roster entry."""
    key, name, units, trap = entry["key"], entry["name"], entry["units"], entry["trap"]
    data = ensure_and_load(session, key, trap)
    if data is None:
        return
    T_b, thr_b, T_f, thr_f, T_mb, thr_mb = data

    fig_state = plot_state_functions(
        T_b, thr_b, T_f, thr_f, trap_name=name,
        T_mb=T_mb, thermo_mb=thr_mb, units_energy=units,
    )
    fig_coef = plot_thermal_coefficients(
        T_b, thr_b, T_f, thr_f, trap_name=name,
        T_mb=T_mb, thermo_mb=thr_mb, units_energy=units,
    )

    if save:
        for tag, fig in (("state", fig_state), ("coefficients", fig_coef)):
            fig.savefig(fig_dir / f"{key}_{tag}.pdf", bbox_inches="tight")
            fig.savefig(fig_dir / f"{key}_{tag}.png", dpi=200, bbox_inches="tight")
        print(f"  {key}: wrote {key}_state.* and {key}_coefficients.*")

    if show:
        plt.show()
    else:
        plt.close(fig_state); plt.close(fig_coef)

## Per-trap figures

Each cell renders both figures for one trap and saves them to `session/figures/`.

### 1. Box — $\nu = 3/2$ (SI, J)

In [ ]:
plot_trap(session, TRAPS[0], fig_dir)

### 2. 2D box + 1D harmonic — $\nu = 2$ (SI, J)

In [ ]:
plot_trap(session, TRAPS[1], fig_dir)

### 3. 2D harmonic + 1D box — $\nu = 5/2$ (SI, J)

In [ ]:
plot_trap(session, TRAPS[2], fig_dir)

### 4. Harmonic oscillator — $\nu = 3$ (eV)

In [ ]:
plot_trap(session, TRAPS[3], fig_dir)

### 5. Quadrupole — $\nu = 9/2$ (eV)

In [ ]:
plot_trap(session, TRAPS[4], fig_dir)

## Reading the figures

- **MB underlay (blue)** is the classical reference. At the high-$T$ start of
  each run all three statistics coincide; departures grow as $T$ falls.
- **Bosons (green)** approach the BE cliff: $\Omega, P, S$ and the heat
  capacities bend away from MB as $\alpha \to 0^-$; $\kappa_T$ grows.
- **Fermions (red)** stiffen with degeneracy pressure: $C_V$ is suppressed,
  $\kappa_T$ shrinks.
- For the **mixed traps**, $P$ and $\kappa_T$ carry the scale of the chosen
  literal $V_g$; every other quantity ($\Omega, S, H, F, G, C_V, C_P, B_P$) is
  independent of $V_g$. With the default geometry these traps sit near the
  classical limit, so their curves hug the MB lines.

Next: the cross-trap dimensionless overlay (one figure), in its own notebook.